In [1]:
import pandas as pd

RAW_DATA_FOLDER = "../../data/raw"
FINAL_DATA_FOLDER = "../../data/final"

In [2]:
# CPI SET
cpi_set = pd.read_csv(f"{RAW_DATA_FOLDER}/ConsumerPriceIndex.csv")
total_expense_cpi = (
    cpi_set[cpi_set['División del gasto'] == "Total"]
    .rename(columns={'Fecha (Mes año)': 'Date', 'Variación anual (%)': 'InflationYOY'})
    .copy()
)
total_expense_cpi["MonthlyDate"] = pd.to_datetime(total_expense_cpi["Date"]).dt.to_period("M")
total_expense_cpi = total_expense_cpi.drop(columns=[
    "Fecha sistema", "Date", "Índice", "Variación mensual (%)", "Variación año corrido (%)", "Orden_División", "División del gasto"
])

total_expense_cpi = total_expense_cpi[["MonthlyDate", "InflationYOY"]]

total_expense_cpi

,MonthlyDate,InflationYOY
0,1954-07,NaN
1,1954-08,NaN
2,1954-09,NaN
3,1954-10,NaN
4,1954-11,NaN
...,...,...
1917,2026-04,5.68131
1930,2026-05,5.84208
1943,2026-06,6.14275
1956,2026-07,6.02629


In [3]:
# TRM SET
trm_set = pd.read_csv(f"{RAW_DATA_FOLDER}/COP_TRM.csv")
trm_monthly_avg = (
    trm_set.rename(columns={
        'Periodo(MMM DD, AAAA)': 'Period', 
        'Tasa Representativa del Mercado (TRM)': 'TRMAvg'
    })
    .assign(Period=lambda df: pd.to_datetime(df['Period']).dt.to_period("M"))
    .groupby('Period')['TRMAvg']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Period': 'MonthlyDate'})
)

trm_monthly_avg

,MonthlyDate,TRMAvg
0,1991-11,694.18
1,1991-12,630.41
2,1992-01,644.06
3,1992-02,635.67
4,1992-03,640.50
...,...,...
414,2026-05,3714.02
415,2026-06,3500.98
416,2026-07,3268.95
417,2026-08,3130.87


In [4]:
# OCCUPATION & UNEMPLOYEMENT SET
occ_unem_set = pd.read_csv(f"{RAW_DATA_FOLDER}/OccupationAndUnemployement.csv")
occ_unem_total_set = (
    occ_unem_set[occ_unem_set['Cobertura geográfica'] == "Total nacional"]
    .rename(columns={
        'Fecha (Mes año)': 'Date', 'Tasa de ocupación (%)': 'OccupationRate', 'Tasa de desempleo (%)': 'UnemploymentRate'
    })
    .drop(columns=["Fecha sistema", "Metodología", "Cobertura geográfica"])
    .reset_index(drop=True)
    .copy()
)
occ_unem_total_set["MonthlyDate"] = pd.to_datetime(occ_unem_total_set["Date"]).dt.to_period("M")
occ_unem_total_set = occ_unem_total_set.drop(columns=["Date"])

occ_unem_total_set

,OccupationRate,UnemploymentRate,MonthlyDate
0,57.575200,16.622300,2001-01
1,56.928200,17.434200,2001-02
2,57.574500,15.811900,2001-03
3,55.756700,14.515100,2001-04
4,56.225800,14.035800,2001-05
...,...,...,...
302,59.306212,8.793386,2026-03
303,59.055998,8.781457,2026-04
304,59.733833,8.020651,2026-05
305,59.362215,8.026926,2026-06


In [5]:
# POLICY RATE SET
policy_rate_set = pd.read_excel(f"{RAW_DATA_FOLDER}/policy_rate.xlsx")
policy_rate_monthly_last = (
    policy_rate_set.rename(columns={
        'Fecha (dd/mm/aaaa)': 'Date', 
        'Tasa de política monetaria (%)': 'PolicyRate'
    })
    .assign(Date=lambda df: pd.to_datetime(df['Date'], dayfirst=True))
    .sort_values('Date')
    .assign(Period=lambda df: df['Date'].dt.to_period("M"))
    .groupby('Period')
    .last()
    .reset_index()
    .rename(columns={'Period': 'MonthlyDate'})
)
policy_rate_monthly_last = policy_rate_monthly_last.drop(columns=["Date"])

policy_rate_monthly_last


,MonthlyDate,PolicyRate
0,1999-04,19.00
1,1999-05,18.00
2,1999-06,18.00
3,1999-07,17.00
4,1999-08,16.00
...,...,...
325,2026-05,11.25
326,2026-06,11.25
327,2026-07,12.00
328,2026-08,12.00


In [6]:
# COLCAP SET
colcap_set = pd.read_csv(f"{RAW_DATA_FOLDER}/stock_market_colcap.csv")
colcap_monthly_set = (
    colcap_set.rename(columns={'Fecha (DD/MM/AAAA)': 'Date', 'Índice': 'ColcapIndex'})
    .assign(Date=lambda df: pd.to_datetime(df['Date'], dayfirst=True))
    .sort_values("Date")
    .set_index("Date")["ColcapIndex"]
    .resample("ME")
    .last()
    .dropna()
    .reset_index()
)
colcap_monthly_set["MonthlyReturn"] = colcap_monthly_set["ColcapIndex"].pct_change() * 100
colcap_monthly_set["MonthlyDate"] = colcap_monthly_set["Date"].dt.to_period("M")
colcap_monthly_set = colcap_monthly_set.drop(columns=["Date"])

colcap_monthly_set


/tmp/ipykernel_38158/2275464488.py:5: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  .assign(Date=lambda df: pd.to_datetime(df['Date'], dayfirst=True))


,ColcapIndex,MonthlyReturn,MonthlyDate
0,937.3271,NaN,2008-01
1,914.9020,-2.392452,2008-02
2,889.2841,-2.800070,2008-03
3,985.9506,10.870148,2008-04
4,1019.7793,3.431075,2008-05
...,...,...,...
220,2176.9000,-0.050505,2026-05
221,2269.0800,4.234462,2026-06
222,2392.1000,5.421581,2026-07
223,2425.0800,1.378705,2026-08


In [7]:
# UNIFICATION
monthly_dataset = (
    total_expense_cpi
    .merge(trm_monthly_avg, on="MonthlyDate", how="left")
    .merge(occ_unem_total_set, on="MonthlyDate", how="left")
    .merge(policy_rate_monthly_last, on="MonthlyDate", how="left")
    .merge(colcap_monthly_set, on="MonthlyDate", how="left")
)

monthly_dataset

,MonthlyDate,InflationYOY,TRMAvg,OccupationRate,UnemploymentRate,PolicyRate,ColcapIndex,MonthlyReturn
0,1954-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1954-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1954-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1954-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1954-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
861,2026-04,5.68131,3617.52,59.055998,8.781457,11.25,2178.00,-4.741494
862,2026-05,5.84208,3714.02,59.733833,8.020651,11.25,2176.90,-0.050505
863,2026-06,6.14275,3500.98,59.362215,8.026926,11.25,2269.08,4.234462
864,2026-07,6.02629,3268.95,59.519311,8.104578,12.00,2392.10,5.421581


In [8]:
# EXPORTATION
monthly_dataset.to_csv(f"{FINAL_DATA_FOLDER}/economics_monthly.csv", index=False)